In [1]:
import os
import sys

import pandas as pd
import shutil
import random
random.seed(42)

from collections import defaultdict

In [2]:
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
print("GPU Available:", gpus)
print("cuDNN Enabled:", tf.test.is_built_with_cuda())

if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError as e:
        print(e)

2025-07-06 08:25:32.997423: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-07-06 08:25:33.004079: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1751801133.011539   92174 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1751801133.013870   92174 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1751801133.019934   92174 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

GPU Available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
cuDNN Enabled: True


In [3]:
def list_files(directory):
    all_files = []
    for root, dirs, files in os.walk(directory):
        for file in files:
            all_files.append(os.path.join(root, file))
    return all_files

def split_by_prefix(crop_list, allowed_prefixes, train_ratio=0.7, val_ratio=0.20, test_ratio=0.10):
    # Group crops by prefix
    prefix_to_crops = defaultdict(list)
    for crop in crop_list:
        prefix = os.path.basename(crop).split('_')[0] + '_'
        if prefix in allowed_prefixes:
            prefix_to_crops[prefix].append(crop)

    train, val, test = [], [], []
    for prefix, crops in prefix_to_crops.items():
        random.shuffle(crops)
        n = len(crops)
        n_train = int(n * train_ratio)
        n_val = int(n * val_ratio)
        n_test = n - n_train - n_val
        train.extend(crops[:n_train])
        val.extend(crops[n_train:n_train+n_val])
        test.extend(crops[n_train+n_val:])
    return train, val, test

In [4]:
IMAGES_PATH = '../../media/data/input_color/'

sys.path.insert(0, "../../")
from config import MEDIA_PATH, CROPPED_PATH, MODELS_PATH

#Configuration
BATCH_SIZE = 32

# Paths
TRAIN_IMAGES = os.path.join(CROPPED_PATH, 'classification1', 'train')
VALID_IMAGES = os.path.join(CROPPED_PATH, 'classification1', 'valid')

TRAIN_CSV = os.path.join(CROPPED_PATH, 'onion_cell_merged', 'data_v2', 'train')
VALID_CSV = os.path.join(CROPPED_PATH, 'onion_cell_merged', 'data_v2', 'valid')
TEST_CSV = os.path.join(CROPPED_PATH, 'onion_cell_merged', 'data_v2', 'test')

In [5]:
train_crops = list_files(TRAIN_IMAGES) #Paths to the ina_crops made from SAM detection of the full_images
valid_crops = list_files(VALID_IMAGES) #Paths to the ina_crops made from SAM detection of the full_images

train_csv = list_files(TRAIN_CSV) #Paths to the ina_crops made from SAM detection of the full_images
valid_csv = list_files(VALID_CSV) #Paths to the ina_crops made from SAM detection of the full_images
test_csv = list_files(TEST_CSV) #Paths to the ina_crops made from SAM detection of the full_images

all_crops = sorted(train_crops + valid_crops)
all_csv = sorted(train_csv + valid_csv + test_csv)

allowed_prefixes = ('A_', 'B_', 'C_', 'D_', 'E_', 'F_')
all_crops = [img for img in all_crops if os.path.basename(img).startswith(allowed_prefixes)]
print(f"Total crops found: {len(all_crops)}")

Total crops found: 25565


In [6]:
# Prepare output lists
nd_crops = []
d_crops = []

# Build a lookup for all crop paths: {(image, cell_id): crop_path}
crop_lookup = {}
for crop_path in all_crops:
    base = os.path.basename(crop_path)
    # Example: 'A_6_1.png' -> image='A_6', cell_id='1'
    parts = base.split('_')
    if len(parts) >= 3:
        image = '_'.join(parts[:2])
        cell_id = os.path.splitext(parts[2])[0]
        crop_lookup[(image, cell_id)] = crop_path

# Loop through each CSV and assign crops to the correct class list
for csv_path in all_csv:
    df = pd.read_csv(csv_path)
    for _, row in df.iterrows():
        image = str(row['image'])
        cell_id = str(row['cell_id'])
        assigned_class = row['assigned_class']
        key = (image, cell_id)
        crop_path = crop_lookup.get(key)
        if crop_path:
            if assigned_class == 'nd':
                nd_crops.append(crop_path)
            elif assigned_class == 'd':
                d_crops.append(crop_path)

print(f"ND crops: {len(nd_crops)}")
print(f"D crops: {len(d_crops)}")

ND crops: 12058
D crops: 2711


In [7]:
# Split nd_crops
nd_train, nd_val, nd_test = split_by_prefix(nd_crops, allowed_prefixes)
# Split d_crops
d_train, d_val, d_test = split_by_prefix(d_crops, allowed_prefixes)

print(f"ND train: {len(nd_train)}, val: {len(nd_val)}, test: {len(nd_test)}")
print(f"D train: {len(d_train)}, val: {len(d_val)}, test: {len(d_test)}")

ND train: 8438, val: 2410, test: 1210
D train: 1895, val: 539, test: 277


In [8]:
def copy_images_to_folder(image_paths, image_class, subset):
    """
    Copies images to a folder named after the subset (train/val/test) inside output_base.
    Creates the folder if it does not exist.
    """
    output_path = os.path.join(CROPPED_PATH, 'classification_v2', subset, image_class)
    os.makedirs(output_path, exist_ok=True)
    for img_path in image_paths:
        shutil.copy(img_path, output_path)


copy_images_to_folder(nd_train, 'nd', 'train')
copy_images_to_folder(nd_val, 'nd', 'valid')
copy_images_to_folder(nd_test, 'nd', 'test')
copy_images_to_folder(d_train, 'd', 'train')
copy_images_to_folder(d_val, 'd', 'valid')
copy_images_to_folder(d_test,  'd', 'test')